# V024: residual temporal-convolution/LSTM comparison

Two controlled 200-epoch A fits add zero-initialized nonlinear temporal adapters with spans3 and7 to the unchanged47-feature deep curve-loss LSTM. Train-only statistics, original grouped split, seed, Adam optimizer, objective and complete rate schedule remain fixed. Known lung/phase/time/step channels remain available unchanged at every step.

The first-place/third-place supplied reports motivate this independently implemented hybrid. Actual CUDA shape, native-metric, zero-adapter control, exact resume and bounded-efficiency proofs are required before approved-data fitting. Three predeclared replacement ensembles use the unchanged weighted-median/training-only-grid policy. Only the lowest eligible candidate can advance to new cold F confirmation; no new weights or policies are searched after results.

Default inert. The approved operational runner sets the reviewed source pin and PA3_RUN_V024 only after all proofs. Modeling remains in notebooks; no test predictions or submission are produced by this A screen.

In [ ]:
import os,json,hashlib,gc,time,fcntl,copy
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import pandas as pd
V024_PLAN_SHA='4b18cf799c974bbfae02bce20dc65bdc42f9668c17d5fd809839090c0e2ccbe5'
V024_SCREEN_NOTEBOOK='controlled_experiments_v024.ipynb'
V024_COMPONENT='convolution_components_v024.ipynb'

def v24_require(c,m):
    if not bool(c):raise ValueError(m)
def v24_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def v24_code(p):return '\n'.join(''.join(c['source']) for c in json.loads(Path(p).read_text())['cells'] if c['cell_type']=='code')
def v24_json(p,v,resume=False):
    p=Path(p)
    if p.exists():v24_require(resume and json.loads(p.read_text())==v,'Existing immutable artifact changed: '+str(p));return
    p.parent.mkdir(parents=True,exist_ok=True);t=p.with_name(p.name+'.tmp');t.write_text(json.dumps(v,indent=2,allow_nan=False)+'\n');os.replace(t,p)
def v24_array(p,a):
    p=Path(p)
    if p.exists():v24_require(np.array_equal(np.load(p),a),'Existing predictions changed');return
    t=p.with_name(p.name+'.tmp')
    with t.open('wb') as f:np.save(f,a,allow_pickle=False)
    os.replace(t,p)
def v24_event(out,event,**kw):
    with (out/'events.jsonl').open('a') as f:f.write(json.dumps(dict(event=event,time_utc=datetime.now(timezone.utc).isoformat(),**kw))+'\n')

def run_screen_v024(root='.',resume=False):
    root=Path(root).resolve();out=root/'outputs/controlled_experiments_v024';out.mkdir(parents=True,exist_ok=True);locks=[];began=time.monotonic()
    try:
        for p in [root/'outputs/.v011_gpu.lock',out/'.run.lock']:
            f=p.open('a+');fcntl.flock(f,fcntl.LOCK_EX|fcntl.LOCK_NB);locks.append(f)
        v24_require(v24_sha(root/'research/protocol_v024.json')==V024_PLAN_SHA,'Frozen V024 plan changed')
        plan=json.loads((root/'research/protocol_v024.json').read_text())
        for p,h in plan['input_hashes'].items():v24_require(v24_sha(root/p)==h,'Frozen input changed: '+p)
        for flag in ['PA3_RUN_V013','PA3_RUN_V014','PA3_RUN_V019','PA3_RUN_V021','PA3_RUN_FINAL_V011','PA3_RUN_FINAL_V018']:v24_require(os.environ.get(flag,'0')!='1','Disable other notebook run flags')
        namespace={'__name__':'ventilator_v024_components'}
        for name,h in plan['component_code_hashes'].items():
            source=v24_code(root/'research'/name);v24_require(hashlib.sha256(source.encode()).hexdigest()==h,'Component changed: '+name);exec(compile(source,name,'exec'),namespace)
        helper=v24_code(root/'controlled_experiments_v013.ipynb');v24_require(hashlib.sha256(helper.encode()).hexdigest()=='56cb9d7e824e58126bddd71c7c0f7063844974c31ee00a6dac23f2112657615a','Pure scoring helper changed');exec(compile(helper,'controlled_experiments_v013.ipynb','exec'),namespace)
        globals().update({k:v for k,v in namespace.items() if not k.startswith('__')})
        source=hashlib.sha256(v24_code(root/V024_SCREEN_NOTEBOOK).encode()).hexdigest();v24_require(source==os.environ.get('PA3_V024_SOURCE_SHA256'),'Screen source differs from reviewed pin')
        torch.set_num_threads(4);v24_require(torch.cuda.is_available(),'Actual course CUDA required');seed_sequence(plan['seed']);v24_require(torch.are_deterministic_algorithms_enabled(),'Strict deterministic execution required')
        sources=dict(plan['component_code_hashes'],**{V024_SCREEN_NOTEBOOK:source,'protocol_v024.json':V024_PLAN_SHA,'controlled_experiments_v013.ipynb':'56cb9d7e824e58126bddd71c7c0f7063844974c31ee00a6dac23f2112657615a'})
        design=dict(protocol_sha256=V024_PLAN_SHA,source_code_sha256=source,source_fingerprints=sources,runtime=dict(torch=str(torch.__version__),cuda=torch.version.cuda,numpy=np.__version__,pandas=pd.__version__),strict_determinism=True)
        v24_json(out/'design.json',design,resume)
        v24_require((out/'course_proofs.json').is_file(),'Run and independently review actual-CUDA synthetic shape/mask/resume proofs before approved-data training')
        proofs=json.loads((out/'course_proofs.json').read_text());v24_require(proofs['status']=='passed' and proofs['source_code_sha256']==source and proofs['component_code_sha256']==sources[V024_COMPONENT] and proofs['actual_cuda'] and proofs['exact_resume'] and proofs['zero_adapter_control'] and proofs['native_inference'],'Actual-CUDA proofs missing or changed')
        v24_require(proofs['source_fingerprints']==sources and all(proofs['checks'].values()),'Actual CUDA proof source/implementation checks differ')
        profiles=proofs['efficiency_profiles'];gate=plan['GPU_efficiency_gate']
        v24_require(len(profiles)==3 and all(p['step_time_ratio_to_baseline']<=gate['maximum_step_time_ratio_to_same_batch_backbone'] and p['peak_allocated_bytes']<gate['maximum_peak_allocated_bytes'] for p in profiles[1:]),'Actual full-model GPU efficiency gate failed')
        old_csv=root/plan['preserved_official_CSV'];v24_require(v24_sha(old_csv)==plan['preserved_official_CSV_sha256'],'Accepted official artifact changed')
        if (out/'screen_summary.json').exists():
            done=json.loads((out/'screen_summary.json').read_text())
            v24_require(resume and done['status']=='complete' and done['protocol_sha256']==V024_PLAN_SHA and done['source_code_sha256']==source and done['source_fingerprints']==sources,'Completed screen identity changed or explicit resume missing')
            for record in done['records'].values():
                for key,hkey in [('checkpoint_file','checkpoint_sha256'),('report_file','report_sha256'),('prediction_file','prediction_sha256')]:v24_require(v24_sha(root/record[key])==record[hkey],'Completed screen artifact changed')
            for candidate in done['candidates']:v24_require(v24_sha(root/candidate['prediction_file'])==candidate['prediction_sha256'],'Completed candidate changed')
            return done
        frame=pd.read_csv(root/'data/train.csv',dtype=V013_DTYPES);bids=_validate_feature_frame_v005(frame)
        labels=frame.pressure.to_numpy(copy=True).reshape(-1,80);uo=frame.u_out.to_numpy(copy=True).reshape(-1,80);ids=frame.id.to_numpy(copy=True).reshape(-1,80)
        known=frame.drop(columns='pressure');del frame;gc.collect();x,names,xbids=make_sequence_features(known,'combined');del known;gc.collect()
        v24_require(x.shape==(67905,80,47) and np.array_equal(xbids,bids),'Approved feature/group schema changed')
        with np.load(root/'outputs/improvements_v005/sequence/split_breath_ids.npz') as z:
            fit=np.sort(pd.Index(bids).get_indexer(z['A_fit']));val=np.sort(pd.Index(bids).get_indexer(z['A_validation']))
        v24_require(len(fit)==61114 and len(val)==6791 and not np.intersect1d(fit,val).size and np.array_equal(np.sort(np.r_[fit,val]),np.arange(67905)),'Original grouped A split changed')
        mask=uo[val]==0;truth=labels[val][mask].astype(np.float64);row_ids=ids[val][mask];counts=mask.sum(1)
        v24_require(np.array_equal(row_ids,np.load(root/'outputs/improvements_v005/sequence/A_validation_row_ids.npy')),'Original A row alignment changed');v24_array(out/'A_row_ids.npy',row_ids)
        grid=pressure_codebook_v016(labels[fit],uo[fit]);v24_require(np.array_equal(grid,np.load(root/'outputs/postprocessing_screen_v016/A_fit_pressure_codebook.npy')),'Training-only pressure grid changed')
        predictions={k:np.load(root/p) for k,p in plan['reference_prediction_files'].items()}
        reference=selected_postprocessing_v016(predictions,plan['reference_weights'],grid)
        v24_require(np.array_equal(reference,np.load(root/plan['reference_prediction_file'])) and abs(np.abs(reference-truth).mean()-plan['reference_A_MAE'])<1e-15,'Frozen reference score/recipe changed')
        records={}
        for name,arm in plan['arms'].items():
            stage=out/name;stage.mkdir(exist_ok=True);checkpoint=stage/'model.pt';report_path=stage/'report.json';record_path=stage/'record.json';factory=namespace[arm['factory']]
            v24_require(arm['feature_mode']=='combined' and arm['raw_feature_count']==47,'Frozen full-feature arm changed')
            selected_names=names;xx=x
            recipe=dict(model_spec=plan['base_model_spec'],train_cfg=plan['train_cfg'],seed=plan['seed'],feature_names=selected_names,loss_kind=arm['loss_kind'],curve_aux_weight=arm.get('curve_aux_weight'),adapter_span=arm['adapter_span'],factory=arm['factory'],protected_output_channels=arm['protected_output_channels'],activation=arm['activation'],initialization=arm['initialization'])
            identity=dict(member=name,recipe=recipe,sources=sources,train_group_sha256=v013_ah(bids[fit]),validation_group_sha256=v013_ah(bids[val]))
            if record_path.exists():
                record=json.loads(record_path.read_text());v24_require(resume and record['identity']==identity and v24_sha(checkpoint)==record['checkpoint_sha256'] and v24_sha(report_path)==record['report_sha256'],'Completed A member changed')
            else:
                if report_path.exists():
                    v24_require(resume,'Existing report requires explicit resume');report=json.loads(report_path.read_text());v24_require(report['completed'] and v24_sha(checkpoint)==report['checkpoint_sha256'],'Incomplete or changed saved report')
                else:
                    v24_event(out,'A_fit_started',member=name,feature_count=len(selected_names),seed=plan['seed'])
                    common=dict(epoch_log_path=stage/'epochs.jsonl',feature_names=selected_names,device='cuda',resume=resume and _last_path(checkpoint).exists(),arm='v024_A_'+name,source_fingerprints=sources)
                    args=(xx[fit],labels[fit],uo[fit],xx[val],labels[val],uo[val],plan['base_model_spec'],plan['train_cfg'],plan['seed'],checkpoint)
                    report=fit_validate_v008(*args,**common,model_factory=factory,loss_kind='curve_l1')
                    v24_require(report['completed'] and report['executed_epochs']==200,'Screen fit incomplete; resume exact saved state');v24_json(report_path,report)
                common=dict(device='cuda',batch_size=512,expected_arm='v024_A_'+name,source_fingerprints=sources)
                p=predict_checkpoint_v008(checkpoint,xx[val],**common,model_factory=factory,expected_loss_kind='curve_l1')
                p=p[mask].astype(np.float64);v24_require(p.shape==truth.shape and np.isfinite(p).all(),'New native predictions invalid');v24_array(stage/'A_predictions.npy',p)
                record=dict(identity=identity,identity_sha256=v013_sig(identity),checkpoint_file=str(checkpoint.relative_to(root)),checkpoint_sha256=v24_sha(checkpoint),report_file=str(report_path.relative_to(root)),report_sha256=v24_sha(report_path),prediction_file=str((stage/'A_predictions.npy').relative_to(root)),prediction_sha256=v24_sha(stage/'A_predictions.npy'),prediction_array_sha256=v013_ah(p),row_ids_array_sha256=v013_ah(row_ids),native_A_MAE=float(np.abs(p-truth).mean()),best_epoch=report['best_epoch'],selected_lr_schedule=report['selected_lr_schedule'],executed_epochs=report['executed_epochs'],training_breaths=61114,elapsed_seconds=report['elapsed_seconds'])
                v24_json(record_path,record)
            predictions[name]=np.load(root/record['prediction_file']);records[name]=record
            if xx is not x:del xx
            gc.collect();torch.cuda.empty_cache()
        candidates=[]
        for name,weights in plan['candidates'].items():
            p=selected_postprocessing_v016({k:predictions[k] for k in weights},weights,grid);error=np.abs(p-truth);interval=v013_interval(error,np.abs(reference-truth),counts,plan['bootstrap_seed']);gain=float(np.abs(reference-truth).mean()-error.mean())
            path=out/(name+'_A_predictions.npy');v24_array(path,p)
            candidates.append(dict(name=name,weights=weights,MAE=float(error.mean()),gain=gain,interval=interval,eligible=bool(gain>=plan['minimum_A_gain'] and interval['high']<0),prediction_file=str(path.relative_to(root)),prediction_sha256=v24_sha(path)))
        eligible=[c for c in candidates if c['eligible']];selected=min(eligible,key=lambda c:(c['MAE'],c['name'])) if eligible else None
        result=dict(status='complete',protocol_sha256=V024_PLAN_SHA,source_code_sha256=source,source_fingerprints=sources,reference_A_MAE=plan['reference_A_MAE'],records=records,candidates=candidates,selected_for_confirmation=selected,no_F_fit_performed=True,no_final_fit_performed=True,no_submission=True,elapsed_seconds=time.monotonic()-began,completed_utc=datetime.now(timezone.utc).isoformat())
        v24_require(v24_sha(old_csv)==plan['preserved_official_CSV_sha256'],'Accepted CSV changed during screen')
        v24_json(out/'screen_summary.json',result);return result
    finally:
        for f in reversed(locks):fcntl.flock(f,fcntl.LOCK_UN);f.close()

if os.environ.get('PA3_RUN_V024','0')=='1':
    V024_SCREEN_RESULT=run_screen_v024(Path.cwd(),resume=os.environ.get('PA3_RESUME_V024','0')=='1')
    print(json.dumps(V024_SCREEN_RESULT['candidates'],indent=2))
